# CYMEK FORMATION-DIAG-001 — the cheap diagnostic (T4 x2)

## BEFORE RUNNING

```
Kaggle Settings:
  Accelerator -> GPU T4 x2
  Internet    -> ON
```

Nonofficial diagnostic: fresh seeds, canonical M0 control, NO sealed data.
Two arms run side by side (one process per GPU, no DDP):

- GPU0: full six-family mixture, 3000 updates, seed 73012
- GPU1: identity-only, 2000 updates, fresh seed 91002

Measurements S5 lacked: teacher-forced token accuracy (overall / by position /
by answer length), CE by family, target rank + margin, full-vocab vs
shared-only decoding rescue, grad norm pre-clip, clipping multiplier, tied
input/output gradient decomposition. Final model checkpoints are exported
separately with an EXPORT_VERIFIED receipt. The FORMATION-BASELINE-GATE-001
decision tree is applied verbatim and the mapped world is printed.


In [ ]:
# CELL 1 — checkout the diagnostic commit + verify blobs + hardware info
import hashlib, json, subprocess, sys
from pathlib import Path
REPO = Path('/kaggle/working/An-Ra-the-new-AGI')
BRANCH = 'cymek-next-core-architecture'
DIAG_COMMIT = 'aba4b1eb9b285893886f72cc9f026da764812980'
BLOBS = {"anra_v5/formation_diag.py": "4daa43fddaf8342694df7396dea9879e0e6c202d04c390f92b24cee50f8ed6b7", "anra_v5/formation_mux_model.py": "1928937192ad1c0709f45a41a9fa33365714d8b65552366d5167c84e575d8e79", "anra_v5/formation_mux_train.py": "281f384ea9049974d29b47af7b2e65865b3403d8cd852c0e3912dd7aca787b21", "v5_experiments/formation_mux_protocol.py": "981fa001e90aca43c2c46a076317f7557713341ddd6500214af5f160d32358ec", "v5_experiments/formation_mux_data.py": "758367ce2ab3dec9307d1c0a8b8c56424c8353cdb3927f974a369a1941ae40f5", "tools/formation_diag_001.py": "12a122436719407aaeb0ff1cdc8672885d57979988b8f021f29bc04552d300f9", "docs/cymek/experiments/FORMATION-DIAG-001/PREREGISTRATION.json": "b179130a6bd579de0b3170c186a83104797fbf8a12ab486476fa17da4b47ded5"}
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                    'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', DIAG_COMMIT], check=True)
head = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
assert head == DIAG_COMMIT
for rel, expected in BLOBS.items():
    assert hashlib.sha256((REPO / rel).read_bytes()).hexdigest() == expected, rel
print('diagnostic commit +', len(BLOBS), 'blob hashes verified')
try:
    import tokenizers
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tokenizers'], check=True)
import torch
print('torch', torch.__version__, '| devices', torch.cuda.device_count())


In [ ]:
# CELL 2 — generate the surface, launch both diagnostic arms (one per GPU)
import json, subprocess, sys, threading
from pathlib import Path
REPO = Path('/kaggle/working/An-Ra-the-new-AGI')
OUT = Path('/kaggle/working/FORMATION_DIAG_001')
OUT.mkdir(parents=True, exist_ok=True)
SURFACE = OUT / 'SURFACE_MANIFEST.json'
if not SURFACE.exists():
    sys.path.insert(0, str(REPO))
    from v5_data.corpus_loading import _load_tokenizer
    from v5_experiments.formation_mux_data import build_surface
    tokenizer, _ = _load_tokenizer(REPO.resolve())
    surface = build_surface(seed=73011, tokenizer=tokenizer)
    SURFACE.write_text(json.dumps(surface, indent=2), encoding='utf-8')
print('surface ready')

def worker(gpu, variant, seed, updates):
    env = {'CUDA_VISIBLE_DEVICES': str(gpu), 'OMP_NUM_THREADS': '2',
           'MKL_NUM_THREADS': '2'}
    subprocess.run([sys.executable, 'tools/formation_diag_001.py',
                    '--variant', variant, '--seed', str(seed),
                    '--surface', str(SURFACE), '--out', str(OUT),
                    '--device', 'cuda', '--updates', str(updates), '--progress'],
                   cwd=str(REPO), env=env, check=True)

t0 = threading.Thread(target=worker, args=(0, 'SIX_FAMILY', 73012, 3000))
t1 = threading.Thread(target=worker, args=(1, 'IDENTITY_ONLY', 91002, 2000))
t0.start(); t1.start(); t0.join(); t1.join()
print('both diagnostic arms complete')


In [ ]:
# CELL 3 — apply the preregistered decision tree, package, print outputs
import hashlib, json, shutil, sys
from pathlib import Path
REPO = Path('/kaggle/working/An-Ra-the-new-AGI')
sys.path.insert(0, str(REPO))
from tools.formation_diag_001 import apply_decision_tree
OUT = Path('/kaggle/working/FORMATION_DIAG_001')
full = json.loads((OUT / 'DIAG_SIX_FAMILY_RECEIPT.json').read_text())
ident = json.loads((OUT / 'DIAG_IDENTITY_ONLY_RECEIPT.json').read_text())
decision = apply_decision_tree(full_receipt=full, identity_receipt=ident,
    gate_prereg=REPO / 'docs/cymek/experiments/FORMATION-BASELINE-GATE-001/PREREGISTRATION.json')
print(json.dumps(decision, indent=1))
(OUT / 'DIAG_DECISION.json').write_text(json.dumps(decision, indent=1))
archive = shutil.make_archive('/kaggle/working/FORMATION_DIAG_001_RESULTS', 'zip', OUT)
print('BUNDLE:', archive, hashlib.sha256(Path(archive).read_bytes()).hexdigest()[:16])
